# Practical Core — 6 Indicators

| Aspect                          | Includes                                                                      |
| ------------------------------- | ----------------------------------------------------------------------------- |
| Monetary & Financial Conditions | Interest rates, central banks, liquidity, credit, stock/bond markets, banking |
| Inflation & Prices              | CPI, PPI, wages, commodity-driven price pressures                             |
| Real Economic Activity          | GDP, recession, manufacturing, services, investment, productivity             |
| Labor & Consumption             | Employment, wages, retail sales, consumer spending/confidence                 |
| Fiscal & Government             | Taxes, deficits, spending, regulation, stimulus                               |
| External Sector                 | Trade, exports/imports, FX, tariffs, current account                          |



In [1]:
from google.colab import userdata

token = userdata.get("Github_Token")
!git clone https://{token}@github.com/Nas-Mohd/economic-news-sentiment.git
!git config --global user.email "anasamohammad.school@gmail.com"
!git config --global user.name "Nas-Mohd"

fatal: destination path 'economic-news-sentiment' already exists and is not an empty directory.


In [2]:
!pip install snorkel # Run once


In [3]:

import sys
sys.path.append('/content/economic-news-sentiment')

In [6]:
!git -C /content/economic-news-sentiment/ pull

Already up to date.


In [5]:
!git -C /content/economic-news-sentiment add .
!git -C /content/economic-news-sentiment commit -m "made semantic ABSENT threshold lower"
!git -C /content/economic-news-sentiment push

[main 084f4cd] made semantic ABSENT threshold lower
 6 files changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (12/12), 13.33 KiB | 593.00 KiB/s, done.
Total 12 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Nas-Mohd/economic-news-sentiment.git
   71c1eae..084f4cd  main -> main


In [12]:
!pip install snorkel.preprocessing

ERROR: Could not find a version that satisfies the requirement snorkel.preprocessing (from versions: none)
ERROR: No matching distribution found for snorkel.preprocessing


In [7]:
# RUN EVERYTIME CHANGES ARE MADE
import importlib
from labeling.labeling_functions import make_keyword_lf, make_semantic_lf
importlib.reload(labeling)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'labeling' is not defined

In [8]:
import importlib
import labeling
from labeling.labeling_functions import make_keyword_lf, make_semantic_lf
from labeling.labeling_functions.lf import embedder
from labeling.anchors import ANCHORS, SEMANTIC_THRESHOLD
from labeling.keywords import KEYWORDS
from labeling.labeling_functions import lf_lemmatized_cost_push, lf_is_not_economic_news, lf_monetary_policy_inflation_exclusion, lf_central_banks_nlp, lf_rate_actions_nlp, lf_labor_market_action_nlp, lf_central_bank_fiscal_exclusion, lf_fiscal_sector_clash_exclusion, lf_monetary_exclusion_clash
from labeling.labeling_functions import lf_inflation_exclusion_clash, lf_activity_exclusion_clash, lf_labor_exclusion_clash, lf_fiscal_exclusion_clash, lf_external_exclusion_clash, lf_deny_fiscal_if_pure_labor, lf_advanced_context_regex, lf_fiscal_policy_semantic_false, lf_fiscal_policy_semantic_true
from labeling.labeling_functions import lf_fiscal_actions_nlp
importlib.reload(labeling)

<module 'labeling' (namespace) from ['/content/economic-news-sentiment/labeling']>

In [9]:
import re
from snorkel.preprocess import preprocessor
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.lf.nlp import nlp_labeling_function
from snorkel.labeling.model import LabelModel
from sentence_transformers import SentenceTransformer, util
from typing import List, Dict
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
import numpy as np

In [16]:
embedder

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [10]:
ECONOMIC_ASPECTS = ["monetary_policy", "inflation", "economic_growth", "labor_market", "consumer_activity", "business_activity", "financial_markets", "trade_external", "fiscal_policy", "energy_commodities", "banking_credit", "corporate_climate"]

# --- Convert to dict of list of tensors ---
ANCHORS_EMBEDDING: Dict[str, List] = {}  # Value type will be list[tensor]
for category in ECONOMIC_ASPECTS:
    ANCHORS_EMBEDDING[category] = [
        embedder.encode(phrase, convert_to_tensor=True) for phrase in ANCHORS[category]
    ]


In [11]:
# Instantiating the merged, simplified keyword and semantic similarity LFs
lf_monetary_policy_keywords = make_keyword_lf("monetary_policy", KEYWORDS)
lf_monetary_policy_semantic = make_semantic_lf("monetary_policy", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_inflation_keywords = make_keyword_lf("inflation", KEYWORDS)
lf_inflation_semantic = make_semantic_lf("inflation", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_economic_growth_keywords = make_keyword_lf("economic_growth", KEYWORDS)
lf_economic_growth_semantic = make_semantic_lf("economic_growth", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_labor_market_keywords = make_keyword_lf("labor_market", KEYWORDS)
lf_labor_market_semantic = make_semantic_lf("labor_market", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_consumer_activity_keywords = make_keyword_lf("consumer_activity", KEYWORDS)
lf_consumer_activity_semantic = make_semantic_lf("consumer_activity", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_business_activity_keywords = make_keyword_lf("business_activity", KEYWORDS)
lf_business_activity_semantic = make_semantic_lf("business_activity", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_financial_markets_keywords = make_keyword_lf("financial_markets", KEYWORDS)
lf_financial_markets_semantic = make_semantic_lf("financial_markets", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_trade_external_keywords = make_keyword_lf("trade_external", KEYWORDS)
lf_trade_external_semantic = make_semantic_lf("trade_external", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_fiscal_policy_keywords = make_keyword_lf("fiscal_policy", KEYWORDS)
lf_fiscal_policy_semantic = make_semantic_lf("fiscal_policy", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_energy_commodities_keywords = make_keyword_lf("energy_commodities", KEYWORDS)
lf_energy_commodities_semantic = make_semantic_lf("energy_commodities", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_banking_credit_keywords = make_keyword_lf("banking_credit", KEYWORDS)
lf_banking_credit_semantic = make_semantic_lf("banking_credit", ANCHORS_EMBEDDING, embedder, SEMANTIC_THRESHOLD)
lf_central_banks_keywords = make_keyword_lf("central_banks", KEYWORDS)
lf_rate_actions = make_keyword_lf("rate_actions", KEYWORDS)

In [12]:
# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab', quiet=True)
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# Embedder model instantiation
#embedder = SentenceTransformer('all-mpnet-base-v2')
#embedder = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
#embedder = SentenceTransformer('sentence-transformers/all-distilroberta-v1')

In [13]:
import json

# Load JSON file
with open("/content/economic-news-sentiment/labeling/data/predictions_review.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten into rows
rows = []

for aspect_combo, samples in data.items():
    for sample in samples:
        rows.append({
            "aspects": aspect_combo,
            "article_id": sample.get("article_id"),
            "title": sample.get("title"),
            "sentence_idx": sample.get("sentence_idx"),
            "text": sample.get("text")
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Optional: inspect
print(df.head())

                               aspects  article_id  \
0                                 None         164   
1                                 None           3   
2                      External_Sector          97   
3                      External_Sector          65   
4  External_Sector + Fiscal_Government         159   

                                               title  sentence_idx  \
0         Where experience meets academic excellence            14   
1  ECB urges banks to brace quickly for AI-assist...             8   
2  China's durian boom is ripening into a regiona...            21   
3  Johor eyeing 12 million visitors, RM42.48bil i...             7   
4              Cautious optimism on growth prospects             4   

                                                text  
0  “I needed a university that was recognised, ap...  
1  (Reporting by Ludwig Burger and Reinhard Becke...  
2  As Southeast Asia's main producing areas enter...  
3  He added that Singapore remains

In [ ]:
df.to_csv("benchmarks.csv", index=False)

In [14]:
# When extracting your final training labels from the matrix:
def filter_high_confidence_solo_votes(label_matrix, probabilities, threshold=0.5):
    final_labels = []

    for row_votes, prob in zip(label_matrix, probabilities):
        # Count how many active (non-abstain) votes are present
        # (Assuming ABSTAIN is coded as -1 or 0 depending on your setup)
        active_votes = sum(1 for vote in row_votes if vote != -1)

        # Guardrail: If only 1 LF voted, don't trust a runaway high probability
        if active_votes == 1 and prob > 0.90:
            final_labels.append(-1) # Force to ABSTAIN/Ignore for training
        else:
            final_labels.append(1 if prob >= threshold else 0)

    return final_labels

In [26]:
ASPECTS = [ # 6 MAIN ASPECTS TO LOOK FOR IN TEXT
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

monetary_lfs = [
    lf_is_not_economic_news,
    lf_monetary_policy_keywords,
    lf_monetary_policy_semantic,
    lf_financial_markets_keywords,
    lf_financial_markets_semantic,
    lf_banking_credit_keywords,
    lf_banking_credit_semantic,
    lf_central_banks_keywords,
    lf_central_banks_nlp,
    lf_rate_actions_nlp,
    lf_monetary_policy_inflation_exclusion,
    lf_monetary_exclusion_clash
]

inflation_lfs = [
    lf_is_not_economic_news,
    lf_inflation_keywords,
    lf_inflation_semantic,
    lf_monetary_policy_inflation_exclusion,
    lf_rate_actions_nlp,
    lf_inflation_exclusion_clash,
    lf_lemmatized_cost_push
]

activity_lfs = [
    lf_is_not_economic_news,
    lf_economic_growth_keywords,
    lf_economic_growth_semantic,
    lf_business_activity_keywords,
    lf_business_activity_semantic,
    lf_trade_external_keywords,
    lf_activity_exclusion_clash,
    lf_advanced_context_regex  # Highly critical here to catch physical "supply shocks"
]

labor_lfs = [
    lf_is_not_economic_news,
    lf_labor_market_keywords,
    lf_labor_market_semantic,
    lf_labor_exclusion_clash,
    lf_consumer_activity_keywords,
    lf_labor_market_action_nlp    # Catches structural labor-actions via dependency parsing
]

fiscal_lfs = [
    lf_is_not_economic_news,
    lf_fiscal_policy_keywords,
    lf_fiscal_policy_semantic,
    lf_central_bank_fiscal_exclusion,
    lf_deny_fiscal_if_pure_labor,
    lf_fiscal_actions_nlp
]

external_lfs = [
    lf_is_not_economic_news,
    lf_trade_external_keywords,
    lf_trade_external_semantic,
    lf_energy_commodities_keywords,
    lf_energy_commodities_semantic,
    lf_external_exclusion_clash,
    lf_advanced_context_regex  # Highly critical here to catch "global oil prices" and geopolitical boundaries
]

# 2. Final Aspect Mapping Dictionary
aspect_lf_groups = {
    "Monetary_Financial": monetary_lfs,
    "Inflation_Prices": inflation_lfs,
    "Real_Economic_Activity": activity_lfs,
    "Labor_Consumption": labor_lfs,
    "Fiscal_Government": fiscal_lfs,
    "External_Sector": external_lfs
}

In [27]:
# ── 1. Load the CSV ──────────────────────────────────────────────────────────
#df = pd.read_csv("malaysian_economic_news_1000.csv")   # ← update path as needed


In [ ]:
# ── 2. Explode each article into one row per sentence ────────────────────────
def explode_to_sentences(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for article_id, row in df.iterrows():
        title = row.get("title", "")
        text  = row.get("text", "")

        # headline as its own sentence row
        if pd.notna(title) and str(title).strip():
            rows.append({
                "article_id":   article_id,
                "title":        title,
                "sentence_idx": -1,           # -1 flags it as the headline
                "text":         str(title).strip(),
                "is_headline":  True,
            })

        # body sentences
        if pd.notna(text) and str(text).strip():
            sentences = sent_tokenize(str(text))
            for i, sent in enumerate(sentences):
                rows.append({
                    "article_id":   article_id,
                    "title":        title,
                    "sentence_idx": i,
                    "text":         sent.strip(),
                    "is_headline":  False,
                })

    return pd.DataFrame(rows)

In [ ]:
sentence_df = explode_to_sentences(df)
print(f"\nTotal sentences: {len(sentence_df)}")
print(f"Total articles:  {sentence_df['article_id'].nunique()}")


Total sentences: 3475
Total articles:  182


In [ ]:
sample_df = sentence_df.sample(n=500, random_state=42)

In [ ]:
sample_df.to_csv("sample_df.csv", index=False)

In [28]:
# We'll store the LF matrices for each aspect
L_matrices = {}

for aspect, lf_list in aspect_lf_groups.items():
  applier = PandasLFApplier(lf_list)
  L_train = applier.apply(df)
  L_matrices[aspect] = L_train

# Train a LabelModel per aspect
label_models = {}
preds_proba = {}  # shape: (num_samples, 1) probability of class 1


100%|██████████| 47/47 [00:00<00:00, 435.16it/s]


In [17]:
# Train a LabelModel per aspect
label_models = {}
preds_proba = {}  # shape: (num_samples, 1) probability of class 1

for aspect, L in L_matrices.items():
    label_model = LabelModel(cardinality=2, verbose=False)
    label_model.fit(L, n_epochs=20, seed=42)
    label_models[aspect] = label_model
    # Get probabilistic predictions for class 1
    preds_proba[aspect] = label_model.predict_proba(L)[:, 1]  # probability of presence

100%|██████████| 20/20 [00:00<00:00, 487.91epoch/s]


In [ ]:
for aspect, L in L_matrices.items():
    # 1. Add precise class balance priors if you know roughly how much data is fiscal
    # 2. Add L2 regularization (prec_init) to prevent the model from tanking the weights of low-coverage LFs
    label_model = LabelModel(
        cardinality=2,
        verbose=True
    )

    # Use fit with a specified optimizer learning rate and a higher penalty on structural assumptions
    label_model.fit(
        L,
        n_epochs=100,
        seed=42,
        lr=0.01,              # Lower learning rate avoids aggressive weight drops
        l2=0.1                # Adds slight regularization to stabilize the matrix
    )

    label_models[aspect] = label_model
    preds_proba[aspect] = label_model.predict_proba(L)[:, 1]

100%|██████████| 100/100 [00:00<00:00, 949.42epoch/s]


In [29]:
import numpy as np

for aspect, L in L_matrices.items():
    # 1. Initialize the LabelModel
    label_model = LabelModel(
        cardinality=2,
        verbose=True
    )

    # 2. Fit with a specified optimizer learning rate and regularization penalty
    label_model.fit(
    L,
    n_epochs=100,
    seed=42,
    lr=0.01,
    l2=0.1,
)

    label_models[aspect] = label_model

    # 3. Get raw continuous probability scores for the PRESENT class (col 1)
    raw_probs = label_model.predict_proba(L)[:, 1]

    # --------------------------------------------------------------------------
    # GUARDRAIL: Identify and filter out runaway high-confidence solo votes
    # --------------------------------------------------------------------------
    # Count how many labeling functions voted (anything that isn't an ABSTAIN / -1)
    active_vote_counts = (L != -1).sum(axis=1)

    # Create a mask for rows where exactly ONE labeling function voted
    is_solo_vote = (active_vote_counts == 1)

    # Create a mask for rows where the model generated a runaway high probability
    is_runaway_confidence = (raw_probs > 0.80)

    # Combine masks: True if it's a dangerous solo voter with runaway confidence
    should_filter = is_solo_vote & is_runaway_confidence

    # Apply guardrail: Force dangerous rows to 0.0 probability (or map to a specific fallback index)
    sanitized_probs = np.where(should_filter, 0.0, raw_probs)

    # Store the sanitized probabilities for your final downstream extraction
    preds_proba[aspect] = sanitized_probs

100%|██████████| 100/100 [00:00<00:00, 634.72epoch/s]


In [30]:
for aspect, lf_list in aspect_lf_groups.items():
  print(f'\nASPECT: {aspect}')
  print(LFAnalysis(L=L_matrices[aspect], lfs=lf_list).lf_summary())



ASPECT: Monetary_Financial
                                         j Polarity  Coverage  Overlaps  \
lf_is_not_economic_news                  0      [0]  0.212766  0.063830   
lf_monetary_policy_nlp_matcher           1      [1]  0.170213  0.170213   
lf_monetary_policy_semantic              2      [1]  0.127660  0.085106   
lf_financial_markets_nlp_matcher         3      [1]  0.170213  0.170213   
lf_financial_markets_semantic            4   [0, 1]  0.085106  0.042553   
lf_banking_credit_nlp_matcher            5      [1]  0.170213  0.170213   
lf_banking_credit_semantic               6   [0, 1]  0.127660  0.085106   
lf_central_banks_nlp_matcher             7      [1]  0.170213  0.170213   
lf_central_banks_nlp                     8      [1]  0.042553  0.042553   
lf_rate_actions_nlp                      9       []  0.000000  0.000000   
lf_monetary_policy_inflation_exclusion  10      [1]  0.085106  0.063830   
lf_monetary_exclusion_clash             11      [0]  0.085106  0.000000 

In [31]:
# Define the order you want (same as aspect_lf_groups)
aspect_order = [
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

# Stack probabilities into an (n_samples, 6) array
probs_matrix = np.column_stack([preds_proba[aspect] for aspect in aspect_order])

# Or as a tidy DataFrame (each row is a sample, each column an aspect)
probs_df = pd.DataFrame(probs_matrix, columns=aspect_order)

# If you later need crisp 0/1 presence labels, threshold (e.g., at 0.5):
#presence_matrix = (probs_matrix >= 0.5).astype(int)
#presence_df = pd.DataFrame(presence_matrix, columns=aspect_order)

In [32]:

# ── 1. Concatenate ────────────────────────────────────────────────────────────
# probs_df columns are your label model probability outputs
#combined_df = pd.concat([sample_df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)
combined_df = pd.concat([df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)

print(combined_df.head())
print(f"\nShape: {combined_df.shape}")

# ── 2. Check for predictions significantly close to 0 or 1 ───────────────────
THRESHOLD = 0.15  # "close to 0" = < 0.15, "close to 1" = > 0.85

label_cols = probs_df.columns.tolist()

# Boolean mask: any label column is near 0 or 1
near_boundary = combined_df[label_cols].apply(
    lambda col: (col < THRESHOLD) | (col > (1 - THRESHOLD))
)

# Rows where AT LEAST ONE label is near a boundary
flagged = combined_df[near_boundary.any(axis=1)].copy()
flagged["flagged_labels"] = near_boundary[near_boundary.any(axis=1)].apply(
    lambda row: [label_cols[i] for i, v in enumerate(row) if v], axis=1
)

print(f"\nFlagged rows (any label near 0 or 1): {len(flagged)} / {len(combined_df)}")
print(flagged[["article_id", "text", "flagged_labels"] + label_cols].head(10))

# ── 3. Summary: per-label breakdown ──────────────────────────────────────────
print("\n── Per-label flagged counts ──")
for col in label_cols:
    n_near_zero = (combined_df[col] < THRESHOLD).sum()
    n_near_one  = (combined_df[col] > (1 - THRESHOLD)).sum()
    print(f"  {col:30s}  near 0: {n_near_zero:4d}  |  near 1: {n_near_one:4d}")

                               aspects  article_id  \
0                                 None         164   
1                                 None           3   
2                      External_Sector          97   
3                      External_Sector          65   
4  External_Sector + Fiscal_Government         159   

                                               title  sentence_idx  \
0         Where experience meets academic excellence            14   
1  ECB urges banks to brace quickly for AI-assist...             8   
2  China's durian boom is ripening into a regiona...            21   
3  Johor eyeing 12 million visitors, RM42.48bil i...             7   
4              Cautious optimism on growth prospects             4   

                                                text  Monetary_Financial  \
0  “I needed a university that was recognised, ap...            0.190927   
1  (Reporting by Ludwig Burger and Reinhard Becke...            0.408319   
2  As Southeast Asia's mai

In [33]:
# ── Filter by aspect ──────────────────────────────────────────────────────────
ASPECT = "Fiscal_Government"   # ← change this to any label col
DIRECTION = "near_1"            # ← "near_1", "near_0", or "both"

if DIRECTION == "near_1":
    mask = combined_df[ASPECT] > (1 - THRESHOLD)
elif DIRECTION == "near_0":
    mask = combined_df[ASPECT] < THRESHOLD
else:  # both
    mask = (combined_df[ASPECT] > (1 - THRESHOLD)) | (combined_df[ASPECT] < THRESHOLD)

aspect_flagged = combined_df[mask][["article_id", "text", ASPECT]]

print(f"── {ASPECT} | {DIRECTION} | {len(aspect_flagged)} rows ──")
for _, row in aspect_flagged.iterrows():
    print(f"\n [{row["article_id"]}] [{row[ASPECT]:.4f}]  {row['text'][:120]}")

── Fiscal_Government | near_1 | 0 rows ──


In [34]:
view_df = combined_df[combined_df["article_id"].isin([150, 16])][['article_id',"text", "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"]]

In [35]:
#view_df.iloc[1]
view_df.head(10)

,article_id,text,Monetary_Financial,Inflation_Prices,Real_Economic_Activity,Labor_Consumption,Fiscal_Government,External_Sector
28,16,Teves said the market would be watching for ke...,1.0,0.999978,0.999995,0.999046,0.616745,0.999263
36,150,The ringgit gained against the Thai baht to 12...,0.5,0.500000,0.500000,0.500000,0.500000,0.760015
37,16,"Gold seen bullish, to hit US$5,600 by year-end...",0.0,0.500000,0.500000,0.500000,0.476548,0.500000


In [36]:
chosen = view_df.iloc[2]
print(f": [text]: "+chosen.text)
for aspect, lf_list in aspect_lf_groups.items():
    print(f"\n  {aspect:20s} {chosen[aspect]}")
    print(f"  {'-' * 40}")
    for lf in lf_list:
        result = lf(chosen)
        label  = "PRESENT" if result == 1 else "ABSTAIN" if result == -1 else "ABSENT"
        print(f"    {lf.name:40s}  →  {label}")

print("\n" + "=" * 60)

: [text]: Gold seen bullish, to hit US$5,600 by year-end - UBS

  Monetary_Financial   0.0
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSTAIN
    lf_monetary_policy_nlp_matcher            →  ABSTAIN
    lf_monetary_policy_semantic               →  ABSTAIN
    lf_financial_markets_nlp_matcher          →  ABSTAIN
    lf_financial_markets_semantic             →  PRESENT
    lf_banking_credit_nlp_matcher             →  ABSTAIN
    lf_banking_credit_semantic                →  ABSTAIN
    lf_central_banks_nlp_matcher              →  ABSTAIN
    lf_central_banks_nlp                      →  ABSTAIN
    lf_rate_actions_nlp                       →  ABSTAIN
    lf_monetary_policy_inflation_exclusion    →  ABSTAIN
    lf_monetary_exclusion_clash               →  ABSTAIN

  Inflation_Prices     0.5
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSTAIN
    lf_inflation_nlp_matcher                  →  ABSTA

In [39]:
embedding = embedder.encode(chosen.text, convert_to_tensor=True)
print(f'Similarity between:\n{chosen.text} and:\n')
for anchor in ANCHORS['energy_commodities']:
    anchor_embedding = embedder.encode(anchor, convert_to_tensor=True)
    similarity = util.cos_sim(embedding,anchor_embedding)
    print(f"{anchor}\n{similarity.item():.4f}")



Similarity between:
The ringgit gained against the Thai baht to 12.3084/3274 from 12.3473/3664 and edged up against the Philippine peso to 6.58/6.59 from 6.59/6.60 previously. and:

Brazilian soy exports are projected to surge next month, boosting global port activity.
0.0568
As global coal mining hubs enter peak production season, maritime commodity imports are expected to climb sharply throughout the second quarter.natural gas prices spiked during winter demand
0.0946
electricity and household utility bills
0.0183
Industrial metals like copper and iron ore faced downward price pressure as global warehouse inventories reached multi-month highs
0.0767


In [37]:
lf_central_banks_keywords(chosen)

-1

In [ ]:
KEYWORDS['fiscal_policy']

['government spending',
 'tax revenue',
 'budget deficit',
 'stimulus',
 'public debt',
 'infrastructure',
 'austerity',
 'tax base',
 'social security',
 'payroll tax',
 'budget revision',
 'price control',
 'targeted subsidy',
 'targeted subsidies',
 'diesel price',
 'fuel cost',
 'govt to allocate',
 'government allocation',
 'public spending',
 'subsidy rationalisation',
 'subsidy rationalization',
 'subsidy',
 'economy ministry',
 'ministry of economy',
 'capital allocation',
 'state-backed',
 'allocation',
 'funded by',
 'million allocation',
 'disbursement',
 'package',
 'supply']

In [ ]:
lf_fiscal_policy_keywords(chosen)

1

In [ ]:
results = []
for idx, row in df.iterrows():
    result = lf_fiscal_policy_semantic_true(row)
    if result == 1:  # ABSENT
        results.append({"index": idx, "text": row.text,"article_id": row.article_id})

print(f"ABSENT rows: {len(results)} / {len(sample_df)}\n")
for r in results:
    print(f"  [{r['article_id']}]  {r['text'][:120]}")

ABSENT rows: 5 / 500

  [141]  He urged the government to recognise the wider economic effects of diesel price adjustments and to introduce targeted me
  [103]  Faced with the risk of sovereign default, the country turned to a massive bailout led by the International Monetary Fund
  [103]  Faced with the risk of sovereign default, the country turned to a massive bailout led by the International Monetary Fund
  [54]  At the same time, there is also a valid concern on global growth prospects, which may necessitate easing monetary policy
  [149]  Private investment is projected to stay robust, driven by infrastructure rollouts, data centre developments, and continu


In [ ]:
# 1. Parse the specific sentence with your loaded spacy model
doc = nlp(chosen.text)
print(f"{'TOKEN':<15} | {'LEMMA':<15} | {'DEP TAG':<10} | {'HEAD VERB (LEMMA)':<18}")
print("-" * 65)

for token in doc:
    print(f"{token.text:<15} | {token.lemma_:<15} | {token.dep_:<10} | {token.head.lemma_:<18}")

TOKEN           | LEMMA           | DEP TAG    | HEAD VERB (LEMMA) 
-----------------------------------------------------------------
Fernandes       | Fernandes       | nsubj      | say               
said            | say             | ROOT       | say               
that            | that            | mark       | make              
to              | to              | aux        | achieve           
achieve         | achieve         | advcl      | make              
and             | and             | cc         | achieve           
maintain        | maintain        | conj       | achieve           
growth          | growth          | dobj       | maintain          
,               | ,               | punct      | make              
the             | the             | det        | country           
country         | country         | nsubj      | make              
must            | must            | aux        | make              
make            | make            | ccomp      | s

In [ ]:
# Check how spaCy tokenizes your keyword vs the text
print([token.lemma_ for token in nlp("economy ministry")])
print([token.lemma_ for token in nlp("Economy Ministry")])

# Check what the matcher actually sees in your doc
doc = nlp(chosen.text)
print([(token.text, token.lemma_) for token in doc])

['economy', 'ministry']
['Economy', 'Ministry']
[('Rafizi', 'Rafizi'), ('is', 'be'), ('being', 'be'), ('questioned', 'question'), ('over', 'over'), ('allegations', 'allegation'), ('linked', 'link'), ('to', 'to'), ('a', 'a'), ('semiconductor', 'semiconductor'), ('investment', 'investment'), ('deal', 'deal'), ('involving', 'involve'), ('the', 'the'), ('Economy', 'Economy'), ('Ministry', 'Ministry'), ('and', 'and'), ('Arm', 'Arm'), ('Holdings', 'Holdings'), ('.', '.')]


In [ ]:
matcher_test = PhraseMatcher(nlp.vocab, attr="LEMMA")
matcher_test.add("test", [nlp("economy ministry")])
print(matcher_test(nlp(chosen.text)))

[]


In [ ]:
from spacy.tokens import Token
Token.set_extension("lower_lemma", getter=lambda t: t.lemma_.lower(), force=True)
lf_fiscal_policy_keywords = make_keyword_lf("fiscal_policy")
doc = nlp(chosen.text)
print([(token.text, token.lemma_, token._.lower_lemma) for token in doc])

[('Rafizi', 'Rafizi', 'rafizi'), ('is', 'be', 'be'), ('being', 'be', 'be'), ('questioned', 'question', 'question'), ('over', 'over', 'over'), ('allegations', 'allegation', 'allegation'), ('linked', 'link', 'link'), ('to', 'to', 'to'), ('a', 'a', 'a'), ('semiconductor', 'semiconductor', 'semiconductor'), ('investment', 'investment', 'investment'), ('deal', 'deal', 'deal'), ('involving', 'involve', 'involve'), ('the', 'the', 'the'), ('Economy', 'Economy', 'economy'), ('Ministry', 'Ministry', 'ministry'), ('and', 'and', 'and'), ('Arm', 'Arm', 'arm'), ('Holdings', 'Holdings', 'holdings'), ('.', '.', '.')]


In [ ]:
pattern_doc = nlp("economy ministry")
print([(token.text, token._.lower_lemma) for token in pattern_doc])

AttributeError: [E046] Can't retrieve unregistered extension attribute 'lower_lemma'. Did you forget to call the `set_extension` method?